In [ ]:
#@title Copyright 2019 Google LLC. { display-mode: "form" }
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

<table class="ee-notebook-buttons" align="left"><td>
<a target="_blank"  href="http://colab.research.google.com/github/google/earthengine-community/blob/master/guides/linked/ee-api-colab-setup.ipynb">
    <img src="https://www.tensorflow.org/images/colab_logo_32px.png" /> Run in Google Colab</a>
</td><td>
<a target="_blank"  href="https://github.com/google/earthengine-community/blob/master/guides/linked/ee-api-colab-setup.ipynb"><img width=32px src="https://www.tensorflow.org/images/GitHub-Mark-32px.png" /> View source on GitHub</a></td></table>

# Earth Engine Python API Colab Setup

This notebook demonstrates how to setup the Earth Engine Python API in Colab and provides several examples of how to print and visualize Earth Engine processed data.

## Import API and get credentials

The Earth Engine API is installed by default in Google Colaboratory so requires only importing and authenticating. These steps must be completed for each new Colab session, if you restart your Colab kernel, or if your Colab virtual machine is recycled due to inactivity.

### Import the API

Run the following cell to import the API into your session.

In [4]:
import ee

### Authenticate and initialize

Run the `ee.Authenticate` function to authenticate your access to Earth Engine servers and `ee.Initialize` to initialize it. Upon running the following cell you'll be asked to grant Earth Engine access to your Google account. Follow the instructions printed to the cell.

In [5]:
# Trigger the authentication flow.
ee.Authenticate()

# Initialize the library.
ee.Initialize(project='wide-empire-494911-k3')

## Test the API

Test the API by printing the elevation of Mount Everest.

In [9]:
# 2. Load your CSV Database
df = pd.read_csv("DebrisFlowVolume_Inventory.csv")

# We only need one export per unique watershed
# Drop any rows that might be missing coordinate data
df = df.dropna(subset=['DepositLatitude', 'DepositLongitude'])
unique_watersheds = df.drop_duplicates(subset=['WatershedID'])

print(f"Found {len(unique_watersheds)} unique watersheds to process. Starting exports...")

# 3. Define the Base DEM Dataset (USGS 10m)
dem = ee.Image('USGS/3DEP/10m')
slope_img = ee.Terrain.slope(dem)

# 4. Loop Through Every Fire in the CSV
for index, row in unique_watersheds.iterrows():
    watershed_id = str(row['WatershedID'])
    lat = float(row['DepositLatitude'])
    lon = float(row['DepositLongitude'])

    # Optional: You can use the basin Area_km2 to adjust the box size
    # Here we use a standard 5km (5000m) buffer around the deposit point
    deposit_point = ee.Geometry.Point([lon, lat])
    roi = deposit_point.buffer(5000).bounds()

    # Clip the DEM and Slope to the bounding box
    clipped_slope = slope_img.clip(roi)

    # 5. Create the Export Task
    task_name = f"Export_Slope_{watershed_id}"

    task = ee.batch.Export.image.toDrive(
        image=clipped_slope,
        description=task_name,
        folder='DebrisFlow_PINN_Slopes', # All files will go to this Drive folder
        fileNamePrefix=f"Slope_{watershed_id}",
        region=roi.getInfo()['coordinates'],
        scale=10, # 10-meter resolution
        crs='EPSG:4326',
        maxPixels=1e10
    )

    # 6. Start the Task
    try:
        task.start()
        print(f"Task started: {watershed_id} ({index+1}/{len(unique_watersheds)})")
    except Exception as e:
        print(f"Error starting {watershed_id}: {e}")

    # Brief pause to avoid hitting Earth Engine API rate limits
    time.sleep(1)

print("\nAll tasks have been sent to Google Earth Engine!")
print("You can close this script. The processing will happen on Google's servers.")
print("Check your Google Drive 'DebrisFlow_PINN_Slopes' folder in a few minutes.")

Found 195 unique watersheds to process. Starting exports...


/usr/local/lib/python3.12/dist-packages/ee/deprecation.py:215: DeprecationWarning: 

Attention required for USGS/3DEP/10m! You are using a deprecated asset.
To make sure your code keeps working, please update it.
This dataset has been superseded by USGS/3DEP/10m_collection

Learn more: https://developers.google.com/earth-engine/datasets/catalog/USGS_3DEP_10m

  warnings.warn(warning, category=DeprecationWarning)


Task started: Apple1 (1/195)
Task started: Bush1 (2/195)
Task started: Bush2 (3/195)
Task started: Bush3 (4/195)
Task started: Buzzard1 (5/195)
Task started: Buzzard2 (6/195)
Task started: Buzzard3 (7/195)
Task started: Buzzard4 (8/195)
Task started: Buzzard5 (9/195)
Task started: CameronPeak1 (10/195)
Task started: CameronPeak2 (11/195)
Task started: CameronPeak3 (12/195)
Task started: CameronPeak4 (13/195)
Task started: CameronPeak5 (14/195)
Task started: CameronPeak6 (15/195)
Task started: CameronPeak7 (16/195)
Task started: CameronPeak8 (17/195)
Task started: CameronPeak9 (18/195)
Task started: CameronPeak10 (19/195)
Task started: CameronPeak11 (20/195)
Task started: CameronPeak12 (21/195)
Task started: CameronPeak13 (22/195)
Task started: CameronPeak14 (23/195)
Task started: Carmel1 (24/195)
Task started: Carmel2 (25/195)
Task started: Carmel3 (26/195)
Task started: Carmel4 (27/195)
Task started: Carmel5 (28/195)
Task started: Carmel6 (29/195)
Task started: Carmel7 (30/195)
Task s

In [10]:
# Get dNBR raster files too
